In [70]:
print("Hello World!")

Hello World!


# Task:
1. Load your csvs into this notebook.
2. Clean the files.
3. Output them as clean csvs.

Stretch:
1. Add some data engineering metrics you want to track (e.g. how many rows tracked)
2. Output the files to local SSMS database with the DE metrics.
3. Ask Nirosh

In [71]:
# Loading the data
import pandas as pd
#Load SystemBook
SystemBook = pd.read_csv('03_Library Systembook.csv')
# Load System Customers
SystemCustomers = pd.read_csv('03_Library SystemCustomers.csv')


##Show No of rows and columns
SystembookRawRows = SystemBook.shape[0]
SystembookRawCols = SystemBook.shape[1]

# Output Values
print (SystemBook.head())
print ("Rows:", SystembookRawRows)
print ("Columns:", SystembookRawCols)
print ("-"*50)
SystemCustomersRawRows = SystemCustomers.shape[0]
SystemCustomersRawCols = SystemCustomers.shape[1]

# Output Values
print (SystemCustomers.head())
print ("Rows:", SystemCustomersRawRows)
print ("Columns:", SystemCustomersRawCols)


    Id                                     Books Book checkout Book Returned  \
0  1.0                       Catcher in the Rye   "20/02/2023"    25/02/2023   
1  2.0          Lord of the rings the two towers  "24/03/2023"    21/03/2023   
2  3.0  Lord of the rings the return of the kind  "29/03/2023"    25/03/2023   
3  4.0                                The hobbit  "02/04/2023"    25/03/2023   
4  5.0                                     Dune   "02/04/2023"    25/03/2023   

  Days allowed to borrow  Customer ID  
0                2 weeks          1.0  
1                2 weeks          2.0  
2                2 weeks          3.0  
3                2 weeks          4.0  
4                2 weeks          5.0  
Rows: 114
Columns: 6
--------------------------------------------------
   Customer ID   Customer Name
0          1.0        Jane Doe
1          2.0      John Smith
2          3.0      Dan Reeves
3          NaN             NaN
4          5.0  William Holden
Rows: 9
Columns: 2


In [72]:
# Load System Customers
SystemCustomers = pd.read_csv('03_Library SystemCustomers.csv')
SystemCustomers.head()

,Customer ID,Customer Name
0,1.0,Jane Doe
1,2.0,John Smith
2,3.0,Dan Reeves
3,NaN,NaN
4,5.0,William Holden


In [73]:
## Cleaning Function
def CleanDataBooks(df):
    # Treat blank/whitespace-only cells as NA
    df = df.replace(r'^\s*$', pd.NA, regex=True)
    # Drop duplicate rows
    df = df.drop_duplicates()
    # Drop rows where key columns are missing
    df = df.dropna(subset=['Customer ID', 'Books', 'Book checkout'])
    #Drop rows where CustomerID is not in SystemCustomers
    df = df[df['Customer ID'].isin(SystemCustomers['Customer ID'])]
    
    ###Set Data Types
    #set Customer ID to integer
    df['Customer ID'] = df['Customer ID'].astype(int)
    #set Book checkout to datetime and remove speech marks if they exist
    df['Book checkout'] = pd.to_datetime(
    df['Book checkout'].astype(str).str.replace('"', ''),errors='coerce',dayfirst=True)
    #set Id to integer
    df['Id'] = df['Id'].astype(int)
    #set Books to string
    df['Books'] = df['Books'].astype(str)
    #set Book returned to datetime
    df['Book Returned'] = pd.to_datetime(df['Book Returned'].str.replace('"', ''), errors='coerce', dayfirst=True)
    #set days allowed to borrow to integer and convert 2 weeks to days
    df['Days allowed to borrow'] = df['Days allowed to borrow'].apply(lambda x: 14 if x == '2 weeks' else int(x))
    return df
    #Set book checkout if blank to be book returned - days allowed to borrow
    df['Book checkout'] = df['Book checkout'].fillna(df['Book Returned'] - pd.to_timedelta(df['Days allowed to borrow'], unit='D')) 
    
def CleanDataCustomers(df):
    # Treat blank/whitespace-only cells as NA
    df = df.replace(r'^\s*$', pd.NA, regex=True)
    # Drop duplicate rows
    df = df.drop_duplicates()
    # Drop rows where any value is missing
    df = df.dropna()

    #### Set Data Types
    #set Customer ID to integer
    df['Customer ID'] = df['Customer ID'].astype(int)
    #set Customer Name to string
    df['Customer Name'] = df['Customer Name'].astype(str)
    return df

## Execute the function on both dataframes
SystemBook = CleanDataBooks(SystemBook)
SystemCustomers = CleanDataCustomers(SystemCustomers)

# Show no. of rows and columns after cleaning
NewSystemBookRows = SystemBook.shape[0]
NewSystemBookCols = SystemBook.shape[1]

print("SystemBook - New Rows:", NewSystemBookRows)
print("SystemBook - New Columns:", NewSystemBookCols)

NewSystemCustomersRows = SystemCustomers.shape[0]
NewSystemCustomersCols = SystemCustomers.shape[1]
print("SystemCustomers - New Rows:", NewSystemCustomersRows)
print("SystemCustomers - New Columns:", NewSystemCustomersCols)
print (SystemBook.head())
print (SystemCustomers.head())


##Generally pretty clean wouldn't do more cleaning without more context on the data and what we want to do with it.



SystemBook - New Rows: 18
SystemBook - New Columns: 6
SystemCustomers - New Rows: 8
SystemCustomers - New Columns: 2
   Id                                     Books Book checkout Book Returned  \
0   1                       Catcher in the Rye     2023-02-20    2023-02-25   
1   2          Lord of the rings the two towers    2023-03-24    2023-03-21   
2   3  Lord of the rings the return of the kind    2023-03-29    2023-03-25   
4   5                                     Dune     2023-04-02    2023-03-25   
5   6                              Little Women    2023-04-02    2023-05-01   

   Days allowed to borrow  Customer ID  
0                      14            1  
1                      14            2  
2                      14            3  
4                      14            5  
5                      14            1  
   Customer ID   Customer Name
0            1        Jane Doe
1            2      John Smith
2            3      Dan Reeves
4            5  William Holden
5      

In [74]:
##Get stats of how many rows were dropped and how many columns were dropped for each dataframe
SystemBookRowsDropped = SystembookRawRows - NewSystemBookRows 
SystemBookColsDropped = SystembookRawCols - NewSystemBookCols
SystemCustomersRowsDropped = SystemCustomersRawRows - NewSystemCustomersRows
SystemCustomersColsDropped = SystemCustomersRawCols - NewSystemCustomersCols
##output stats
print("SystemBook - Rows Dropped:", SystemBookRowsDropped)
print("SystemBook - Columns Dropped:", SystemBookColsDropped)
print("SystemCustomers - Rows Dropped:", SystemCustomersRowsDropped)
print("SystemCustomers - Columns Dropped:", SystemCustomersColsDropped)

SystemBook - Rows Dropped: 96
SystemBook - Columns Dropped: 0
SystemCustomers - Rows Dropped: 1
SystemCustomers - Columns Dropped: 0


In [75]:
##Output cleaned dataframes in csvs
SystemBook.to_csv('03_Library Systembook Cleaned.csv', index=False)
SystemCustomers.to_csv('03_Library SystemCustomers Cleaned.csv', index=False)
